# Time and cost of LLM judges

Are typed decision models a cheaper and faster alternative to LLM judges? We compare them with the LLM judges of
CALM that are still served on OpenRouter (October 2026): ChatGPT (`gpt-3.5-turbo-0125`), GPT-4-Turbo (`2024-04-09`)
and GPT-4o (`2024-05-13`). Claude-3.5 Sonnet, Qwen2-72B-Instruct and GLM-4 are no longer served.

Each LLM judge receives prompts exactly as in CALM: the prompt string as one user message, temperature 0.7, no limit
on the length of the reply (it writes an explanation, then `[[A]]` or `[[B]]`). We measure tokens, cost and latency
only, not biases.

* **Sample:** 10 prompts per test (the originals count as one test), taken in turn from the test's versions, so that
  short and long prompts are both covered: 100 prompts per judge.
* **Input tokens of the whole design** are counted exactly with OpenAI's tokenizer (`tiktoken`), so only the length
  of the replies and the latency come from the sample.
* `SMOKE = True` sends 1 prompt per test (10 per judge) to check that everything works.
* Answers are cached: a re-run sends nothing that was answered before.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
SMOKE = False
CONCURRENCY = 8

In [3]:
import sys
from pathlib import Path
CODE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(CODE))   # judgebias and tdm
import numpy as np
import pandas as pd
from judgebias import authority, design, llm_cost, run
pd.set_option("display.width", 250)

rows = design.build_design(citations=authority.load_citations())
picked = llm_cost.sample(rows, per_test=1 if SMOKE else llm_cost.PER_TEST)
print(len(picked), "prompts per judge:", pd.Series([r["bias"] for r in picked]).value_counts().to_dict())

100 prompts per judge: {'authority': 10, 'bandwagon': 10, 'compassion': 10, 'distraction': 10, 'diversity': 10, 'fallacy': 10, 'original': 10, 'position': 10, 'sentiment': 10, 'verbosity': 10}


## Send the sample to each judge

In [4]:
records = {name: llm_cost.run_judge(model, picked, smoke=SMOKE, concurrency=CONCURRENCY)
           for name, model in llm_cost.JUDGES.items()}
for name, recs in records.items():
    bad = [r for r in recs if not r["ok"]]
    print(f"{name}: {len(recs) - len(bad)} of {len(recs)} answered", f"(e.g. {bad[0]['error'][:120]})" if bad else "",
          "| versions reported:", sorted({r["snapshot"] for r in recs if r["ok"]}))

openai/gpt-3.5-turbo:  10%|#         | 10/100 [00:00<?, ?it/s]

openai/gpt-4-turbo:  10%|#         | 10/100 [00:00<?, ?it/s]

openai/gpt-4o-2024-05-13:  10%|#         | 10/100 [00:00<?, ?it/s]

ChatGPT: 100 of 100 answered  | versions reported: ['openai/gpt-3.5-turbo']
GPT-4-Turbo: 100 of 100 answered  | versions reported: ['openai/gpt-4-turbo']
GPT-4o: 100 of 100 answered  | versions reported: ['openai/gpt-4o-2024-05-13']


## One reply per judge, in full

To check that the judges answer as an LLM judge in CALM does: an explanation, then the verdict.

In [5]:
for name, recs in records.items():
    r = next(x for x in recs if x["ok"])
    print("=" * 30, name, "|", r["id"], "=" * 30)
    print(r["reply"])

============================== ChatGPT | 377f49403971/authority/book-on-R1 ==============================
After evaluating the responses provided by Assistant A and Assistant B, it is clear that Assistant A has provided a more detailed and creative explanation in response to the user question. Assistant A used the analogy of toys in a toy box to explain the specific skills needed to become a freelance accountant and a good freelancer, making it easier for a five-year-old to understand. Assistant A listed specific skills required for each role and highlighted that while some skills overlap, there are also unique skills for each. Additionally, Assistant A referenced a source to support their explanation.

On the other hand, Assistant B provided a straightforward response without the use of analogies or creative storytelling. Assistant B simply listed the skills required for each role without providing as much depth or creativity in their explanation. Assistant B did correctly answer the 

## Cost and latency per judgment

Per test, the mean output tokens of the sample; the input tokens are exact for the whole design. The chat format
adds a few tokens per message, estimated as the median difference between the reported and the counted input tokens.
Cost for the full design (14,695 requests per format, as for each typed model) = exact input tokens x input price
+ sum over tests of (requests of the test x mean output tokens of the test) x output price.

In [6]:
full = [r for r, rep in run.requests(rows, "calm")]
n_per_test = pd.Series([r["bias"] for r in full]).value_counts()
summary = []
for name, model in llm_cost.JUDGES.items():
    recs = pd.DataFrame([r for r in records[name] if r["ok"]])
    counted = pd.Series(llm_cost.input_tokens([next(x for x in picked if x["id"] == i) for i in recs.id], model), index=recs.index)
    overhead = float((recs.input_tokens - counted).median())
    price_in, price_out = llm_cost.PRICES[model]
    out_per_test = recs.groupby("bias").output_tokens.mean()
    in_total = sum(llm_cost.input_tokens(full, model)) + overhead * len(full)
    out_total = float((n_per_test * out_per_test.reindex(n_per_test.index)).sum())
    usd_full = (in_total * price_in + out_total * price_out) / 1e6
    summary.append({"judge": name, "kind": "LLM (CALM)", "model": model, "sample": len(recs),
                    "median_latency_s": recs.latency_s.median(), "mean_input_tokens": recs.input_tokens.mean(),
                    "mean_output_tokens": recs.output_tokens.mean(), "chat_overhead_tokens": overhead,
                    "usd_sample": recs.cost.sum(), "usd_per_1000": 1000 * usd_full / len(full), "usd_full_design": usd_full})
llm = pd.DataFrame(summary)
llm.round(4)

,judge,kind,model,sample,median_latency_s,mean_input_tokens,mean_output_tokens,chat_overhead_tokens,usd_sample,usd_per_1000,usd_full_design
0,ChatGPT,LLM (CALM),openai/gpt-3.5-turbo,100,2.0855,796.46,189.82,7.0,0.0683,0.6913,10.1582
1,GPT-4-Turbo,LLM (CALM),openai/gpt-4-turbo,100,11.1102,796.46,291.75,7.0,1.6717,16.7476,246.1063
2,GPT-4o,LLM (CALM),openai/gpt-4o-2024-05-13,100,2.6264,788.24,243.69,7.0,0.7597,7.8012,114.6383


## The typed decision models on the same prompts

From the stored answers of the full run (CALM format): latency and cost of the very same prompts, and the measured
cost of the full design.

In [7]:
ids = {r["id"] for r in picked}
typed = []
for model in [m for m in run.MODELS if "mercury" not in m]:
    recs = [r for r in run.load_records(model, "calm") if r["ok"]]
    same = [r for r in recs if r["id"] in ids and r["repeat"] == 0]
    usd_full = sum(r["cost"] or 0 for r in run.load_records(model, "calm"))
    typed.append({"judge": model.split("/")[1].replace("-experimental", ""), "kind": "typed", "model": model,
                  "sample": len(same), "median_latency_s": float(np.median([r["latency_s"] for r in same])),
                  "mean_input_tokens": float(np.mean([r["input_tokens"] for r in same])),
                  "mean_output_tokens": float(np.mean([r["output_tokens"] for r in same])),
                  "usd_sample": sum(r["cost"] or 0 for r in same), "usd_per_1000": 1000 * usd_full / len(recs),
                  "usd_full_design": usd_full})
both = pd.concat([llm, pd.DataFrame(typed)], ignore_index=True)
if not SMOKE:
    both.to_csv(CODE / "outputs" / "tables" / "llm_cost.csv", index=False)
both[["judge", "kind", "sample", "median_latency_s", "mean_input_tokens", "mean_output_tokens", "usd_per_1000",
      "usd_full_design"]].round(3)

,judge,kind,sample,median_latency_s,mean_input_tokens,mean_output_tokens,usd_per_1000,usd_full_design
0,ChatGPT,LLM (CALM),100,2.086,796.46,189.82,0.691,10.158
1,GPT-4-Turbo,LLM (CALM),100,11.110,796.46,291.75,16.748,246.106
2,GPT-4o,LLM (CALM),100,2.626,788.24,243.69,7.801,114.638
3,jev-1.13,typed,100,0.251,1057.92,32.00,0.045,0.656
4,kev-4b,typed,100,1.914,770.84,52.72,0.033,0.480
5,tev1-4b,typed,100,0.597,849.72,2.00,0.036,0.531
6,solar-decide,typed,100,0.757,1091.71,1.00,0.055,0.811
7,d1,typed,100,0.374,772.58,0.00,0.031,0.454
8,pplx-decider-v1-27b,typed,100,0.214,836.83,1.00,0.034,0.496
9,clef,typed,100,0.530,874.90,0.00,0.211,3.094


In [8]:
t = both.groupby("kind").agg(median_latency=("median_latency_s", "median"), usd_per_1000=("usd_per_1000", "median"))
print(t.round(3))
print("cheapest LLM judge / most expensive typed model: %.1f x" % (llm.usd_per_1000.min() / pd.DataFrame(typed).usd_per_1000.max()))
print("median LLM judge / median typed model: %.0f x cost, %.1f x latency" % (
    llm.usd_per_1000.median() / pd.DataFrame(typed).usd_per_1000.median(),
    llm.median_latency_s.median() / pd.DataFrame(typed).median_latency_s.median()))

            median_latency  usd_per_1000
kind                                    
LLM (CALM)           2.626         7.801
typed                0.507         0.040
cheapest LLM judge / most expensive typed model: 3.3 x
median LLM judge / median typed model: 193 x cost, 5.2 x latency


## Notes

**Smoke run, 2026-10-05** (1 prompt per test, 10 per judge): all 30 answered, each judge reports the requested
version, and every reply is an explanation followed by `[[A]]` or `[[B]]`, as CALM's prompt asks.

**Full sample, 2026-10-05** (100 prompts per judge, 8 parallel requests, $2.50 in total): all 300 answered.
* Replies are long: on average 190 (ChatGPT), 244 (GPT-4o) and 292 (GPT-4-Turbo) output tokens, against 0 to 53 for
  the typed models (most report 0 to 2; Jev 32 and Kev 53).
* **Cost per 1,000 judgments** (full design, input exact, output from the sample): ChatGPT $0.69, GPT-4o $7.80,
  GPT-4-Turbo $16.75; typed models $0.03 (D1) to $0.21 (Clef). Median LLM judge / median typed model: about 190 times.
  Even the cheapest LLM judge costs 3.3 times the most expensive typed model.
* **The full design** (14,695 requests, one format) would cost $10 (ChatGPT), $115 (GPT-4o) and $246 (GPT-4-Turbo),
  against $0.45 to $3.09 per typed model.
* **Latency per request** (median, same 100 prompts): ChatGPT 2.1 s, GPT-4o 2.6 s, GPT-4-Turbo 11.1 s; typed models
  0.21 s (Perplexity) to 1.9 s (Kev). Median 2.6 s vs 0.5 s, about 5 times.
* Prices are OpenRouter list prices on 2026-10-05; the three judges are 2024 models, so newer LLMs may be cheaper.
